# Una escalera de 400 réplicas en 4 minutos

Resultados medidos en **H100**. Este cuadernillo **no calcula nada**: trae
los números y los dibuja, para poder leerlos de un golpe.

## El titular

La escalera de producción —200 temperaturas, 10 000 + 5 000 barridos,
400 cadenas— con tres motores distintos:

| | reloj |
|---|---|
| producción, tal como está | ~15.8 min |
| producción + protocolo de partición | ~11.5 min |
| **kernel de Triton escrito a mano** | **~3.9 min** |

Lo medido fueron 60 temperaturas; la cifra de 200 es el escalado lineal en
barridos, que es exacto por construcción. Las tres filas corrieron en las
**mismas 4 GPU, con el mismo trabajo y la misma física**.


In [ ]:
import json, numpy as np
import matplotlib.pyplot as plt
DATA = json.loads(r'''{
 "ladder4": {
  "n_temps": 60,
  "n_therm": 10000,
  "n_meas": 5000,
  "chains": 400,
  "per_device": 100,
  "devices": 4,
  "arms": {
   "A produccion, 1 ola de 100": {
    "minutes": 4.729766429416667,
    "us": 0.7882944049027779,
    "vs_A": 0.0
   },
   "B produccion, protocolo 53+47": {
    "minutes": 3.453703071200001,
    "us": 0.5756171785333335,
    "vs_A": -26.979415944944375
   },
   "C Triton, 1 ola de 100": {
    "minutes": 1.172768784116666,
    "us": 0.19546146401944434,
    "vs_A": -75.20450953301501
   }
  },
  "C_vs_B": -66.04314962984981
 },
 "ladder1": {
  "n_temps": 200,
  "n_therm": 10000,
  "n_meas": 5000,
  "sweeps": 3000000,
  "replicas": 50,
  "arms": {
   "produccion": {
    "minutes": 6.050278606966667,
    "us": 2.4201114427866663
   },
   "paso Triton": {
    "minutes": 2.522046580566667,
    "us": 1.0088186322266668
   },
   "todo Triton": {
    "minutes": 2.2512707158666654,
    "us": 0.900508286346666
   }
  },
  "recovered": 4.475428030507782
 },
 "width": [
  {
   "L": 4,
   "B": 64,
   "suma": 8,
   "prod": 1.659082109375376,
   "kernel": 0.6722291406252578,
   "change": -59.481864289505005
  },
  {
   "L": 4,
   "B": 65,
   "suma": 2,
   "prod": 1.7320356153847936,
   "kernel": 0.6817084615387505,
   "change": -60.64119839779967
  },
  {
   "L": 4,
   "B": 100,
   "suma": 4,
   "prod": 1.8496923499995432,
   "kernel": 0.6092129999998974,
   "change": -67.06409041481693
  },
  {
   "L": 4,
   "B": 101,
   "suma": 2,
   "prod": 1.860274009901542,
   "kernel": 0.611956683168459,
   "change": -67.1039492079531
  },
  {
   "L": 5,
   "B": 64,
   "suma": 6,
   "prod": 2.355791015624664,
   "kernel": 0.7901434374990579,
   "change": -66.45952751078208
  },
  {
   "L": 5,
   "B": 65,
   "suma": 0,
   "prod": 2.3389897692315853,
   "kernel": 0.7918596923078579,
   "change": -66.14522633983118
  },
  {
   "L": 5,
   "B": 100,
   "suma": 2,
   "prod": 2.957140849999007,
   "kernel": 0.7344681500001116,
   "change": -75.16289594388586
  },
  {
   "L": 5,
   "B": 101,
   "suma": 0,
   "prod": 2.1726718811877808,
   "kernel": 0.7282210891099288,
   "change": -66.48269370928588
  },
  {
   "L": 6,
   "B": 64,
   "suma": 7,
   "prod": 2.9752752343736866,
   "kernel": 0.9247267187495467,
   "change": -68.91962437403856
  },
  {
   "L": 6,
   "B": 65,
   "suma": 1,
   "prod": 2.6806154615384843,
   "kernel": 0.9045234615384218,
   "change": -66.25687367261214
  },
  {
   "L": 6,
   "B": 100,
   "suma": 3,
   "prod": 3.2273495499993032,
   "kernel": 0.8620981000007077,
   "change": -73.28773699146119
  },
  {
   "L": 6,
   "B": 101,
   "suma": 1,
   "prod": 2.6222570297035066,
   "kernel": 0.8523632178222746,
   "change": -67.4950545210799
  },
  {
   "L": 8,
   "B": 64,
   "suma": 9,
   "prod": 3.293707343750008,
   "kernel": 1.164681250003241,
   "change": -64.6392004980865
  },
  {
   "L": 8,
   "B": 65,
   "suma": 3,
   "prod": 3.7950502307703227,
   "kernel": 1.1683875384628313,
   "change": -69.21285707921523
  },
  {
   "L": 8,
   "B": 100,
   "suma": 5,
   "prod": 4.11140605000071,
   "kernel": 1.1252436999996007,
   "change": -72.6311707888982
  },
  {
   "L": 8,
   "B": 101,
   "suma": 3,
   "prod": 3.7823159901014187,
   "kernel": 1.1327788613887533,
   "change": -70.05065509192481
  }
 ],
 "penalty": {
  "L5_produccion": 17.75998018667524,
  "L5_halo": 0.29806350667476167,
  "L6_produccion": 16.967262452887443,
  "L6_halo": 1.704044987150355
 },
 "substep": [
  {
   "B": 64,
   "prod": 2.217973671876372,
   "stencil": 1.1433378125014215,
   "kernel": 0.7889136718741696,
   "change": -64.43088203086015,
   "stencil_pct": -48.451245071174576
  },
  {
   "B": 65,
   "prod": 2.1914737692309947,
   "stencil": 1.1566592307697368,
   "kernel": 0.801126615383597,
   "change": -63.443476867865115,
   "stencil_pct": -47.22002850275422
  },
  {
   "B": 75,
   "prod": 2.0882263333324618,
   "stencil": 1.1180585333344575,
   "kernel": 0.7630525333335451,
   "change": -63.45929935114647,
   "stencil_pct": -46.45893907724925
  },
  {
   "B": 100,
   "prod": 2.860919249999938,
   "stencil": 1.0413240500000143,
   "kernel": 0.7227794500010987,
   "change": -74.73611147881527,
   "stencil_pct": -63.60176715927803
  },
  {
   "B": 101,
   "prod": 2.054250495050799,
   "stencil": 1.0727433168312366,
   "kernel": 0.7216129702979148,
   "change": -64.87220170877602,
   "stencil_pct": -47.779332685291195
  },
  {
   "B": 128,
   "prod": 2.25789980468738,
   "stencil": 1.0355277734364066,
   "kernel": 0.7057295312495881,
   "change": -68.74398368853659,
   "stencil_pct": -54.1375675179802
  }
 ],
 "substep_summary": {
  "mean_change_pct": -66.6,
  "best_change_pct": -74.7,
  "wins": 6,
  "n": 6,
  "dispersion_prod_pct": 39.3,
  "dispersion_halo_pct": 13.5,
  "dispersion_stencil_pct": 11.7,
  "mean_stencil_pct": -51.3,
  "mean_rng_pct": -31.4
 },
 "obs_check": {
  "E_rel": 1.2202676202832663e-07,
  "M_rel": 1.984156341450216e-07,
  "Mz_rel": 1.3339601423467684e-07,
  "Q_abs": 8.381903171539307e-08,
  "ok": true
 },
 "exact": [
  {
   "sublattice": "par",
   "dE_max_abs": 1.430511474609375e-06,
   "dE_rel": 1.4557110114736231e-07,
   "spin_mismatch": 0,
   "spin_total": 182520,
   "mismatch_frac": 0.0
  },
  {
   "sublattice": "impar",
   "dE_max_abs": 1.430511474609375e-06,
   "dE_rel": 1.504606050502403e-07,
   "spin_mismatch": 0,
   "spin_total": 182520,
   "mismatch_frac": 0.0
  }
 ],
 "physics": [
  {
   "T": 20.0,
   "E_prod": -1.4063559770584106,
   "E_halo": -1.406455397605896,
   "E_sd_prod": 0.02657989040017128,
   "E_sigma": 0.003740442341504572,
   "M_prod": 0.47530901432037354,
   "M_halo": 0.4790010452270508,
   "norm_err": 1.7881393432617188e-07
  },
  {
   "T": 5.0,
   "E_prod": -3.065532922744751,
   "E_halo": -3.0667033195495605,
   "E_sd_prod": 0.006291085388511419,
   "E_sigma": 0.18604052123452527,
   "M_prod": 0.8983781933784485,
   "M_halo": 0.8979169726371765,
   "norm_err": 1.7881393432617188e-07
  },
  {
   "T": 1.0,
   "E_prod": -3.429443836212158,
   "E_halo": -3.429527997970581,
   "E_sd_prod": 0.001061942894011736,
   "E_sigma": 0.07925262167809323,
   "M_prod": 0.9702267050743103,
   "M_halo": 0.9701972007751465,
   "norm_err": 1.7881393432617188e-07
  }
 ],
 "control": [
  {
   "therm": 600,
   "meas": 300,
   "ctl_mean": 0.8685219980995825,
   "ctl_max": 2.5337690361987635,
   "k_mean": 0.9907975075155736,
   "k_max": 1.8957184989711207,
   "M_ctl": 0.0003985166549682617,
   "M_k": 0.0005164146423339844
  },
  {
   "therm": 10000,
   "meas": 5000,
   "ctl_mean": 0.8648550387974773,
   "ctl_max": 1.9591366228423455,
   "k_mean": 1.024011999166416,
   "k_max": 3.2860324216759693,
   "M_ctl": 0.00012958049774169922,
   "M_k": 0.00014144182205200195
  }
 ],
 "acceptance": [
  {
   "T": 20.0,
   "prod": 0.4233193116634799,
   "kernel": 0.42179732313575524,
   "ratio": 0.99640463242335
  },
  {
   "T": 10.0,
   "prod": 0.1614225621414914,
   "kernel": 0.16110133843212238,
   "ratio": 0.9980100445370985
  },
  {
   "T": 5.0,
   "prod": 0.072282982791587,
   "kernel": 0.07181644359464627,
   "ratio": 0.993545656544281
  },
  {
   "T": 1.0,
   "prod": 0.013216061185468453,
   "kernel": 0.01359847036328872,
   "ratio": 1.0289351851851851
  },
  {
   "T": 0.1,
   "prod": 0.0028910133843212237,
   "kernel": 0.0028833652007648185,
   "ratio": 0.9973544973544974
  }
 ]
}''')
print("datos cargados:", list(DATA))


## 1 · Cuatro GPU, 400 cadenas

`A` es lo que hacía el proyecto. `B` es la mejor opción **sin tocar el
kernel**: partir el lote en dos olas impares, que esquiva la vectorización de
ancho 4 de XLA. `C` es el kernel, en **una sola ola**, porque no tiene nada
que esquivar.


In [ ]:
l4 = DATA["ladder4"]
esc = 200 / l4["n_temps"]          # a las 200 temperaturas de produccion
names, meas, full = [], [], []
for k, v in l4["arms"].items():
    names.append(k); meas.append(v["minutes"]); full.append(v["minutes"] * esc)

fig, ax = plt.subplots(figsize=(10, 4.2))
y = np.arange(len(names))
b = ax.barh(y, full, color=["#c2c2c2", "#8ab4f8", "#2e7d32"], height=.6)
ax.set_yticks(y); ax.set_yticklabels(names, fontsize=11)
ax.invert_yaxis()
ax.set_xlabel("minutos de la escalera completa (200 T x 10000+5000, "
              f"{l4['chains']} cadenas, {l4['devices']} GPU)", fontsize=11)
for r, m, f_ in zip(b, meas, full):
    ax.text(f_ + .2, r.get_y() + r.get_height() / 2,
            f"{f_:.1f} min   (medido {m:.2f} min a {l4['n_temps']} T)",
            va="center", fontsize=10)
ax.set_xlim(0, max(full) * 1.45)
ax.set_title("La misma escalera, tres motores", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

print(f"{'brazo':<38}{'medido':>9}{'a 200 T':>10}{'vs A':>9}")
for k, v in l4["arms"].items():
    print(f"{k:<38}{v['minutes']:>8.2f}m{v['minutes']*esc:>9.1f}m"
          f"{v['vs_A']:>+8.1f}%")
print(f"\nEl kernel contra el protocolo (la barra real): "
      f"{l4['C_vs_B']:+.1f}%")

## 2 · Una GPU: de dónde sale cada punto

Dos cambios, medidos por separado. Quitar las **seis matrices desplazadas**
que `jnp.roll` materializaba vale la mitad; generar los números aleatorios
**dentro** del kernel vale otro tanto sobre lo que queda.


In [ ]:
s = DATA["substep"]; sm = DATA["substep_summary"]
Bs = [r["B"] for r in s]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(Bs, [r["prod"] for r in s], "o-", label="produccion (6 rolls + Threefry)",
        color="#b00020", lw=2)
ax.plot(Bs, [r["stencil"] for r in s], "s-", label="solo el stencil",
        color="#8ab4f8", lw=2)
ax.plot(Bs, [r["kernel"] for r in s], "^-", label="stencil + su propio RNG",
        color="#2e7d32", lw=2)
ax.set_xlabel("replicas por lanzamiento (B)", fontsize=11)
ax.set_ylabel("us / replica-barrido", fontsize=11)
ax.set_title("El paso de Monte Carlo, por lote", fontsize=13)
ax.legend(fontsize=10); ax.grid(alpha=.25)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

print(f"stencil solo      : {sm['mean_stencil_pct']:+.1f}% de media")
print(f"+ su propio RNG   : {sm['mean_change_pct']:+.1f}% "
      f"(gana {sm['wins']}/{sm['n']}, mejor {sm['best_change_pct']:+.1f}%)")
print(f"\nFijate en B=100 contra B=101 en la curva roja: +39% por una sola")
print(f"replica de diferencia. En la verde no se distingue.")
print(f"\ndispersion entre lotes:  produccion {sm['dispersion_prod_pct']:.1f}%"
      f"   kernel {sm['dispersion_halo_pct']:.1f}%")

## 3 · Los múltiplos de 4 dejan de importar

La regla medida en este proyecto era: **hay castigo si `v₂(B) + v₂(L) ≥ 2`**,
donde `v₂` es cuántas veces entra el 2. Venía de que XLA decidía vectorizar de
cuatro en cuatro el bucle que le entregábamos.

El kernel no le entrega ningún bucle: la forma del bloque la fija el kernel y
el número de capas solo dice cuántos bloques hay. Así que la regla no tiene
por dónde actuar — y se midió sobre `L ∈ {4,5,6,8}` para comprobarlo.

En `L = 4` y `L = 8` **no se puede puntuar** un castigo: `v₂(4) = 2` y
`v₂(8) = 3` ya disparan la regla para *cualquier* `B`, así que no queda grupo
de control. Eso no es un hueco en la medición: es la regla comportándose como
está documentada, y por eso solo `L = 5` y `L = 6` tienen fila.


In [ ]:
w = DATA["width"]; pen = DATA["penalty"]
print(f"{'L':>3}{'B':>6}{'v2(B)+v2(L)':>13}{'produccion':>13}{'kernel':>9}"
      f"{'cambio':>9}")
for r in w:
    mark = " <-" if r["suma"] >= 2 else ""
    print(f"{r['L']:>3}{r['B']:>6}{r['suma']:>13}{r['prod']:>13.3f}"
          f"{r['kernel']:>9.3f}{r['change']:>+8.1f}%{mark}")

print(f"\n=== castigo por v2 >= 2, por motor ===")
ks = sorted(pen)
fig, ax = plt.subplots(figsize=(8, 3.4))
lab = [k.replace("_", "  ") for k in ks]
col = ["#b00020" if "produccion" in k else "#2e7d32" for k in ks]
bb = ax.bar(range(len(ks)), [pen[k] for k in ks], color=col, width=.55)
ax.set_xticks(range(len(ks))); ax.set_xticklabels(lab, fontsize=10)
ax.set_ylabel("castigo (%)", fontsize=11); ax.axhline(0, color="k", lw=.8)
ax.set_title("La regla del ancho, por motor", fontsize=13)
for r, k in zip(bb, ks):
    ax.text(r.get_x() + r.get_width() / 2, r.get_height() + .4,
            f"{pen[k]:+.1f}%", ha="center", fontsize=10)
ax.set_ylim(0, max(pen.values()) * 1.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
for k in ks:
    print(f"  {k:<22}{pen[k]:+7.1f}%")
print("\nMuerta. Con este kernel el numero de replicas se elige por fisica")
print("o por logistica, no para esquivar al compilador.")

## 4 · ¿Y la física es la misma?

Cinco pruebas, y ningún tiempo se miró antes de que pasaran. Las dos primeras
son **exactas** (no hay aleatoriedad involucrada); las tres últimas son
estadísticas.


In [ ]:
e = DATA["exact"]; o = DATA["obs_check"]; p = DATA["physics"]
c = DATA["control"]; a = DATA["acceptance"]

print("1. el stencil contra sim_core.substep, MISMA aleatoriedad")
for r in e:
    print(f"   {r['sublattice']:>5}: dE max |dif| {r['dE_max_abs']:.2e}   "
          f"espines distintos {r['spin_mismatch']}/{r['spin_total']}")
print("\n2. los observables contra los de JAX, mismo estado")
print(f"   E {o['E_rel']:.2e} relativo | |M| {o['M_rel']:.2e} | "
      f"Q {o['Q_abs']:.2e} absoluto")
print("\n3. las dos cadenas a equilibrio, cada una con su RNG")
for r in p:
    print(f"   T={r['T']:>5.1f}  E/sitio {r['E_prod']:+9.5f} / "
          f"{r['E_halo']:+9.5f}  ({r['E_sigma']:.2f} sigma)   "
          f"|norma-1| {r['norm_err']:.1e}")
print("\n4. control: produccion contra SI MISMA, la vara de medir")
print(f"   {'ventana':>14}{'control':>16}{'kernel':>16}")
for r in c:
    print(f"   {r['therm']:>6}+{r['meas']:<7}"
          f"{r['ctl_mean']:>8.2f}/{r['ctl_max']:<7.2f}"
          f"{r['k_mean']:>8.2f}/{r['k_max']:<7.2f}")
print("\n5. tasa de aceptacion")
for r in a:
    print(f"   T={r['T']:>5.1f}  {r['prod']:.4f} / {r['kernel']:.4f}   "
          f"razon {r['ratio']:.4f}")
print("\nLa cuarta es la clave metodologica: correr produccion contra si")
print("misma con otra semilla da el RUIDO PROPIO DE LA VARA. Fue la que")
print("destapo un bug de RNG mio -- claves Philox adyacentes sobre un")
print("contador compartido -- y, tras arreglarlo, la que lo cerro.")

## 5 · Qué cambió en el código

Para actualizar un espín hacen falta sus 6 vecinos. `jnp.roll` los conseguía
**creando seis copias completas de la red**, cada una corrida un paso: seis
escrituras y seis lecturas de toda la red en cada medio paso. HLO le atribuía
el **31.5 %** del tiempo, y otro **17.1 %** al generador aleatorio.

El kernel lee **su parcela más un borde de un sitio**, hace todo en registros
y escribe una vez. Las seis copias nunca existen.

**El detalle que lo hace funcionar** —y que falló tres veces antes, en Pallas—
no es hacer menos lecturas: sigue emitiendo 21. Es la **forma de la
dirección**:

| | la dirección del vecino es… | el compilador emite… |
|---|---|---|
| intentos en Pallas | una por carril, calculada por carril | accesos **dispersos** |
| este kernel | un **vector** desplazado sobre un tramo contiguo | accesos **coalescidos** |

La GPU trae 32 bytes de golpe: ocho hilos pidiendo ocho números seguidos es
**una** lectura; ocho hilos calculando su propia dirección son **ocho**.

Y hay un detalle de física que lo hace legal sin sincronización: el **tablero
de ajedrez** garantiza que todo vecino de un sitio actualizado tiene la
paridad opuesta, así que ningún bloque lo escribe en ese medio paso. El halo
se lee de memoria global **sin barrera**.

### Cómo usarlo

```python
import sim_core as S, triton_halo as TH

geo = S.build_geometry(Rd=18.30, L=5)
ann = TH.build_anneal_full(geo, BY=4, num_warps=4)

k1  = jnp.where(surf_j, P["KanS"], P["Kan1"])   # igual que hoy
par = TH.pack_params(P, B)                      # (B, 6)
sp  = TH.planar(S.init_spins(key, geo, B))      # (B,3,L,N,N)

obs, final, snaps = ann(sp, seed, betas, par, k1, n_therm, n_meas)
resultados = S.reduce_obs(np.asarray(obs), T, geo["n_active"])  # SIN cambios
```

Tres avisos: la imagen necesita `jax-triton`; `pack_params` descarta `Kan1` a
propósito, porque `sim_core._an` lo recibe y no lo usa; y `seed` es un entero,
no una `PRNGKey`, porque el kernel es por contador.


In [ ]:
l1 = DATA["ladder1"]
print(f"Escalera en UNA H100: {l1['n_temps']}T x {l1['n_therm']}+"
      f"{l1['n_meas']} = {l1['sweeps']:,} barridos, {l1['replicas']} replicas")
print()
for k, v in l1["arms"].items():
    print(f"  {k:<34}{v['minutes']:7.2f} min   {v['us']:7.3f} us/replica-barrido")
print(f"\nLos kernels de observables recuperan {l1['recovered']:+.1f} puntos.")
print()
print("Y una enmienda que conviene leer: predije que los 8 puntos entre el")
print("paso suelto (-66.6%) y la escalera eran el bucle de medicion.")
print("Recuperan 4.5 de los 8. El resto esta en los ocho acumuladores, en el")
print("scan sobre 200 temperaturas, y en que la escalera corre con lote 50")
print("mientras el banco del paso corria entre 64 y 128.")